# Prototype LLM API Caller

This notebook is a tiny starting point for calling an LLM API.
It has two simple examples:
1. Send one short text string and print the response.
2. Send a short company description and ask for a business summary plus commercial risks.

In [12]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key is not None)

model_name = "gpt-5-mini"

client = OpenAI(api_key=api_key)

True


In [13]:
# Example 1: send one small text string to the model and print the answer.

sample_text = "Guidewire provides cloud software for property and casualty insurers."

response = client.responses.create(
    model=model_name,
    input=f"Read this short text and explain it in one simple sentence:\n\n{sample_text}"
)

print(response.output_text)

Guidewire makes cloud-based software used by companies that sell property and casualty insurance.


In [17]:
# Example 2: ask for a short business summary and a few possible commercial risks.

company_description = "Verisk sells data, analytics, and software products that help insurers price risk, manage claims, and support underwriting decisions."

prompt = f"""
You are helping with early-stage commercial due diligence.

Company description:
{company_description}

Please provide:
- a 2-bullet business summary
- 3 possible commercial risks

Keep the answer short and practical.
"""

response = client.responses.create(
    model=model_name,
    input=prompt
)

print(response.output_text)

- Business summary
  - Provides proprietary data, analytics and software to insurers and reinsurers to price risk, underwrite policies, and manage/settle claims (models, loss-costs, predictive analytics, workflow SaaS).
  - Enterprise, subscription-led model with high switching costs and regulatory/industry moats (longitudinal data, insurer relationships), but exposed to P&C market cycles and large-account selling dynamics.

- Three commercial risks
  1. Regulatory and data-privacy constraints: tighter data rules (privacy, AI explainability, classification of certain datasets) or limits on third‑party data sharing could reduce model inputs, slow product rollouts, or force costly reengineering—harming accuracy and differentiation.
  2. Technology/competitive disruption: cloud-native insurtechs, open-model AI providers, or hyperscaler analytics could offer lower‑cost or faster alternatives; loss of perceived predictive advantage would pressure pricing and churn among smaller customers.
 

In [ ]:
from pypdf import PdfReader

pdf_path = "Data/Guidewire/10-K.pdf"
reader = PdfReader(pdf_path)

text = ""
for page in reader.pages[:3]:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print(text[:3000])


In [ ]:
prompt = f"""
    You are a financial analyst reviewing a company's 10-K report. 
    Your task is to extract key information and insights from the document.
    The text from the first few pages is below:
    {text}
"""

response = client.responses.create(
    model=model_name,
    input=prompt
)

print(response.output_text)

## Breaking PDF text to chunks
Preparing for **embedding**

In [ ]:
from pypdf import PdfReader
from pathlib import Path

gw_path = Path("Data/Guidewire")

pdf_files = list(gw_path.glob("*.pdf"))

def extract_pdf_chunks(pdf_path, max_chars=3000, overlap=300):
    reader = PdfReader(pdf_path)
    chunks = []

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        i = 0

        while i < len(text):
            chunk = text[i:i + max_chars]

            chunks.append({
                "file_name": pdf_path.name,
                "page_number": page_num,
                "text": chunk
            })

            i += max_chars - overlap

    return chunks


all_chunks = []

for pdf_file in pdf_files:
    pages = extract_pdf_chunks(pdf_file)
    all_chunks.extend(pages)

print(len(all_chunks))

In [33]:
def search_chunks(chunks, query, top_k=5):
    results = []
    query_words = query.lower().split()

    for chunk in chunks:
        text = chunk["text"].lower()
        score = 0

        for word in query_words:
            if word in text:
                score += 1

        if score > 0:
            results.append({
                "file_name": chunk["file_name"],
                "page_number": chunk["page_number"],
                "text": chunk["text"],
                "score": score
            })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [34]:
query = "What products does Guidewire sell to insurance companies?"

results = search_chunks(all_chunks, query)

for result in results:
    print(result["file_name"], result["page_number"])
    print(result["text"][:500])
    print("-" * 50)


10-Q.pdf 31
d billing
functionality, plus pre-integrated document production, analytics, and other capabilities, that increases agility without adding
complexity. Like InsuranceSuite, InsuranceNow is hosted on GWCP and managed by our internal cloud operations team.
InsuranceNow is currently only available in the United States, and is generally suited to mid-market carriers and managing general
agents whose needs are often not as complex as a typical InsuranceSuite customer.
We reach customers directly throu
--------------------------------------------------
10-K.pdf 7
Table of Contents
Item 1. Business
Overview and Purpose
Guidewire is the platform that property and casualty (“P&C”) insurers rely on to engage with customers, innovate, and operate
more efficiently. Founded in 2001, we serve insurers of all sizes, ranging from global carriers to regional and local providers, helping
them navigate a rapidly changing insurance market.
Our platform combines core systems of record with digit